# Step 01 — Building Geometry Cleaning & Volume Calculation

**Input:** `data/input/buildings.gpkg` — raw ALKIS-like 3D building polygons  
**Output:** `data/output/01_building_volumes_filtered.gpkg` — deduplicated buildings with computed volumes and ALKIS/OSM enrichment

**What this notebook does:**
1. Loads raw buildings, keeps only needed columns
2. Calculates area_m² and volume_m³ (area × height)
3. Filters buildings with volume ≥ MIN_BUILDING_VOLUME_M3
4. Deduplicates by gml_id (keeps largest polygon per ID)
5. Merges touching polygons with same function (union-find)
6. Merges overlapping polygons with same function
7. Saves cleaned result

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config loaded. CRS:', TARGET_CRS)

Config loaded. CRS: EPSG:25832


## 1. Load buildings

In [2]:
volumes = gpd.read_file(BUILDINGS_FILE)
print(f'Loaded {len(volumes):,} raw building polygons')
print('Columns:', list(volumes.columns))
volumes.head()

Loaded 4,878,052 raw building polygons
Columns: ['measHeight', 'gml_id', 'externRef', 'function', 'DqDach', 'DqLage', 'DqBoden', 'creationDa', 'GrundrissA', 'LetzteAend', 'Geom2DRef', 'roofType', 'AGS', 'Land', 'Stadt', 'Strasse', 'HausNr', 'Name', 'DachNeig', 'DachOri', 'DachFlaech', 'Firsthoehe', 'Traufhoehe', 'AbsHoehe', 'DachName', 'Eigentum', 'Lizenz', 'geometry']


,measHeight,gml_id,externRef,function,DqDach,DqLage,DqBoden,creationDa,GrundrissA,LetzteAend,...,DachNeig,DachOri,DachFlaech,Firsthoehe,Traufhoehe,AbsHoehe,DachName,Eigentum,Lizenz,geometry
0,6.049,DENILD61000068Eu,http://repository.gdi-de.org/schemas/adv/cityg...,31001_1000,1000,1000,1300,2020-11-23,2017-09-29,2024-09-30,...,70.702,-28.196,198.042,67.154,64.975,61.105,GableRoof,Landesamt fuer Geoinformation und Landesvermes...,"CC-BY-4.0, siehe https://creativecommons.org/l...",MULTIPOLYGON Z (((591513.617 5805837.129 64.97...
1,6.049,DENILD61000068Eu,http://repository.gdi-de.org/schemas/adv/cityg...,31001_1000,1000,1000,1300,2020-11-23,2017-09-29,2024-09-30,...,70.702,-28.196,198.042,67.154,64.975,61.105,GableRoof,Landesamt fuer Geoinformation und Landesvermes...,"CC-BY-4.0, siehe https://creativecommons.org/l...",MULTIPOLYGON Z (((591501.032 5805847.422 67.15...
2,6.049,DENILD61000068Eu,http://repository.gdi-de.org/schemas/adv/cityg...,31001_1000,1000,1000,1300,2020-11-23,2017-09-29,2024-09-30,...,70.702,-28.196,198.042,67.154,64.975,61.105,GableRoof,Landesamt fuer Geoinformation und Landesvermes...,"CC-BY-4.0, siehe https://creativecommons.org/l...",MULTIPOLYGON Z (((591506.516 5805850.358 61.10...
3,8.032,DENILD61000068Es,http://repository.gdi-de.org/schemas/adv/cityg...,31001_1000,1000,1000,1300,2020-11-23,2017-09-29,2024-09-30,...,39.885,-28.418,117.811,69.867,66.685,61.835,GableRoof,Landesamt fuer Geoinformation und Landesvermes...,"CC-BY-4.0, siehe https://creativecommons.org/l...",MULTIPOLYGON Z (((591550.566 5805867.143 69.86...
4,8.032,DENILD61000068Es,http://repository.gdi-de.org/schemas/adv/cityg...,31001_1000,1000,1000,1300,2020-11-23,2017-09-29,2024-09-30,...,39.885,-28.418,117.811,69.867,66.685,61.835,GableRoof,Landesamt fuer Geoinformation und Landesvermes...,"CC-BY-4.0, siehe https://creativecommons.org/l...",MULTIPOLYGON Z (((591547.297 5805865.36 65.412...


## 2. Select and compute columns

Expected input columns: `gml_id`, `measHeight` (building height in m), `function` (ALKIS code), and optionally `Stadt`, `Strasse`, `HausNr`, `Name`.
Adjust the list below if your source data uses different column names.

In [3]:
columns_to_keep = ['gml_id', 'measHeight', 'function', 'Stadt', 'Strasse', 'HausNr', 'Name', 'geometry']
# Keep only columns that exist in this dataset
columns_to_keep = [c for c in columns_to_keep if c in volumes.columns]
volumes = volumes[columns_to_keep].copy()

volumes['area_m2']   = volumes.geometry.area
volumes['volume_m3'] = volumes['area_m2'] * volumes['measHeight'].astype(float)

print(f'Volume range: {volumes["volume_m3"].min():.1f} – {volumes["volume_m3"].max():.1f} m³')

Volume range: 0.0 – 2436075.6 m³


## 3. Filter and deduplicate

In [4]:
# Filter out micro-buildings
volumes_filtered = volumes[volumes['volume_m3'] >= MIN_BUILDING_VOLUME_M3].copy()
print(f'After volume filter (≥ {MIN_BUILDING_VOLUME_M3} m³): {len(volumes_filtered):,}')

# Keep largest polygon per gml_id (LOD2 has multiple roof surface polygons per building)
volumes_filtered = (
    volumes_filtered
    .loc[volumes_filtered.groupby('gml_id')['area_m2'].idxmax()]
    .reset_index(drop=True)
)
print(f'After deduplication by gml_id: {len(volumes_filtered):,}')

After volume filter (≥ 1 m³): 2,748,082
After deduplication by gml_id: 1,361,998


## 4. Load OSM POIs and flag buildings that contain a POI

In [5]:
# OSM POI file is created in notebook 02/03 — skip flagging if not yet available
if OSM_POIS_MODIFIED_FILE.exists():
    pois = gpd.read_file(OSM_POIS_MODIFIED_FILE)
    pois = pois[pois.geometry.notna() & ~pois.geometry.is_empty].copy()
    if pois.crs != volumes_filtered.crs:
        pois = pois.to_crs(volumes_filtered.crs)

    gdf = volumes_filtered.copy()
    gdf['geometry'] = gdf.geometry.buffer(0)

    poi_in_poly = gpd.sjoin(pois[['geometry']], gdf[['geometry']], how='left', predicate='within')
    poi_poly_idx = set(poi_in_poly['index_right'].dropna().astype(int))
    gdf['has_poi'] = gdf.index.isin(poi_poly_idx)
    print(f'Buildings with at least one POI: {gdf["has_poi"].sum():,}')
else:
    gdf = volumes_filtered.copy()
    gdf['geometry'] = gdf.geometry.buffer(0)
    gdf['has_poi'] = False
    print('OSM POI file not found — running without POI flags (run nb 02+03 first for full results)')

Buildings with at least one POI: 7,925


## 5. Merge touching buildings with same function (union-find)

In [6]:
sidx = gdf.sindex
geoms = gdf.geometry.values
funcs = gdf['function'].to_numpy() if 'function' in gdf.columns else np.full(len(gdf), None)
has_poi = gdf['has_poi'].to_numpy()
n = len(gdf)

parent = np.arange(n, dtype=int)
rank   = np.zeros(n, dtype=int)

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:   parent[ra] = rb
    elif rank[ra] > rank[rb]: parent[rb] = ra
    else:                     parent[rb] = ra; rank[ra] += 1

for i, gi in enumerate(geoms):
    for j in sidx.intersection(gi.bounds):
        if j <= i:
            continue
        if funcs[i] != funcs[j]:
            continue
        if has_poi[i] or has_poi[j]:
            continue
        if gi.touches(geoms[j]):
            union(i, j)

gdf['_cluster'] = [find(i) for i in range(n)]
merged = gdf.dissolve(by='_cluster').reset_index(drop=True)
merged['area_m2']   = merged.geometry.area
merged['volume_m3'] = merged['area_m2'] * merged['measHeight'].astype(float)
print(f'After touching-merge: {len(merged):,} buildings')

After touching-merge: 749,142 buildings


## 6. Merge overlapping buildings with same function

In [7]:
def merge_overlaps_keep_attrs(gdf, area_col='area_m2', name_col='Name', function_col='function'):
    gdf = gdf.copy()
    gdf['geometry'] = gdf.geometry.buffer(0)
    gdf = gdf.reset_index(drop=True)

    sidx  = gdf.sindex
    geoms = gdf.geometry.values
    n     = len(gdf)
    funcs = gdf[function_col].to_numpy() if function_col in gdf.columns else None

    parent = np.arange(n, dtype=int)
    rank   = np.zeros(n, dtype=int)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb: return
        if rank[ra] < rank[rb]:   parent[ra] = rb
        elif rank[ra] > rank[rb]: parent[rb] = ra
        else:                     parent[rb] = ra; rank[ra] += 1

    for i, gi in enumerate(geoms):
        for j in sidx.intersection(gi.bounds):
            if j <= i: continue
            if funcs is not None and funcs[i] != funcs[j]: continue
            gj = geoms[j]
            if gi.intersects(gj):
                inter = gi.intersection(gj)
                if not inter.is_empty and inter.area > 0:
                    union(i, j)

    gdf['_cluster'] = [find(i) for i in range(n)]
    out_rows = []
    for _, grp in gdf.groupby('_cluster', sort=False):
        rep = grp.loc[grp[area_col].idxmax()].copy()
        if name_col in grp.columns:
            names = [x for x in grp[name_col].tolist() if x is not None and str(x).strip()]
            uniq  = list(dict.fromkeys(names))
            rep[name_col] = None if not uniq else (uniq[0] if len(uniq) == 1 else uniq)
        rep['geometry'] = grp.geometry.unary_union
        out_rows.append(rep)

    return gpd.GeoDataFrame(out_rows, crs=gdf.crs).reset_index(drop=True)

merged_no_overlaps = merge_overlaps_keep_attrs(merged)
print(f'After overlap-merge: {len(merged_no_overlaps):,} buildings')

After overlap-merge: 663,615 buildings


## 7. Save output

In [8]:
merged_no_overlaps.to_file(VOLUMES_FILTERED_FILE, driver='GPKG')
print(f'Saved {len(merged_no_overlaps):,} buildings → {VOLUMES_FILTERED_FILE}')

Saved 663,615 buildings → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\capacity-pipeline-generic\data\output\01_building_volumes_filtered.gpkg
